# The same number, reported twice, differently

The parent repo shipped one estimand at two masses and two interval definitions in two
places. Both were called "the 90% interval". Nobody was careless; the two call sites were
written eight months apart, and a float carries no memory of how it was summarized.

Two more of the same kind. A posterior mean printed as `2.0134729` beside an interval half a
unit wide — seven digits of which two are knowledge, and a reader who compares two results
on the fifth digit is comparing arithmetic noise. And a function that could not do what was
asked returning `nan`, which is a value, and propagates like one.

This notebook is the three small types that close those off — `Interval`, `Verdict`, and the
typed failures — plus the statistics the acceptance tests are stated in.

In [ ]:
import numpy as np

from axiom.core import (
    AcceptanceRegion,
    DIGITS,
    Assumption,
    Blocked,
    Failure,
    Interval,
    LedgerLine,
    Multiplicity,
    NonEmptyStr,
    Summary,
    Unsupported,
    Unverified,
    Verdict,
    clopper_pearson,
    decimals_for,
    effective_sample_size,
    eti,
    format_interval,
    format_measured,
    hdi,
    interval,
    adjust,
    is_failure,
    mc_standard_error,
    round_to,
    summarize,
    wald,
    z_score,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import AQUA, BLUE, GOOD, ORANGE, annotate, caption, curve_band, density, intervals, lines, mark_x, shade

enable();  # every axiom result renders itself from here on

## Intervals carry their definition

An `Interval` cannot be constructed without `definition` and `mass`. On a symmetric posterior
the choice hardly matters; on a skewed one — which is most of them, since half the parameters
in this package are positive by construction — the two definitions disagree by enough to move
a decision.

In [ ]:
from axiom.display import show

rng = np.random.default_rng(0)
draws = rng.gamma(2.0, 1.0, size=20_000)      # right-skewed, so HDI and ETI differ

h = hdi(draws, 0.9)
e = eti(draws, 0.9)
show(h)
show(e)
print("HDI is narrower on a skewed posterior:", h.width < e.width)

In [ ]:
fig = density(
    {"posterior": draws},
    colors=(BLUE,),
    title="'The 90% interval' is two different intervals",
    subtitle="same draws, same mass — highest-density against equal-tailed on a skewed posterior",
    x_title="value",
)
shade(fig, h.lower, h.upper, text=f"HDI  [{h.lower:.2f}, {h.upper:.2f}]", color=BLUE, alpha=0.13)
mark_x(fig, e.lower, text=f"ETI lower {e.lower:.2f}", color=ORANGE)
mark_x(fig, e.upper, text=f"ETI upper {e.upper:.2f}", color=ORANGE, right=True)
caption(fig, f"The two agree on how much mass they hold and on nothing else: the HDI is "
             f"{h.width:.2f} wide, the ETI {e.width:.2f}. A number reported without its "
             f"definition has silently picked one.")

In [ ]:
try:
    Interval(lower=0.0, upper=1.0)  # no definition, no mass
except Exception as exc:
    print(type(exc).__name__, "- an interval without provenance cannot exist")

i = interval(draws, definition="eti", mass=0.5)
print(i, i.contains(2.0), i.width)

# a frequentist CI is a third kind, and says so
print(wald(estimate=1.2, se=0.3, mass=0.95))

In [ ]:
s: Summary = summarize(draws, definition="hdi", mass=0.95)
print(s)                                      # mean, sd and interval, all at one resolution
print(s.mean, s.median, s.sd, s.n)            # the stored values keep every digit
print(s.to_json()[:160], "...")

## A printed number stops where its uncertainty stops

Printing digits a result does not have is not neutral — a reader takes trailing digits as
precision and compares two results on digits that are arithmetic. So every summary, card, and
report metric in axiom rounds the value to the place its own uncertainty reaches, keeping
`DIGITS` significant digits of that uncertainty.

This is a display rule and only a display rule: `io` still writes seventeen digits, and a
failure's `detail` still carries the number that reproduces the bug.

In [ ]:
print("digits kept of the uncertainty:", DIGITS)

table(
    [[u, format_measured(12.3456789, u), decimals_for(u)] for u in (0.52, 5.2, 52.0, 520.0)],
    headers=("uncertainty", "12.3456789 prints as", "decimals"),
)

# with nothing to round against, nothing is rounded
print("no stated uncertainty ->", format_measured(12.3456789))

In [ ]:
raw = [("arm A", 2.0134729, 0.52), ("arm B", 2.0139981, 0.48), ("arm C", 2.5081164, 0.50)]
fig = intervals(
    [(f"{name}  ({value:.7f})", value, value - 2 * u, value + 2 * u) for name, value, u in raw],
    title="Two of these are the same result",
    subtitle="seven printed digits, and the intervals they came with",
    x_title="estimate",
    highlight="arm C  (2.5081164)",
)
caption(fig, "Printed at seven digits, A and B look like different findings and C looks like "
             "a third. Printed at the resolution their own uncertainty reaches — 2.0 ± 0.5, "
             "2.0 ± 0.5, 2.5 ± 0.5 — the picture is what the numbers actually support.")

In [ ]:
# an interval is its own resolution: both bounds round to the half-width
print(format_interval(h.lower, h.upper), "from a half-width of", round(h.half_width, 4))
print(h)                                       # and the definition and mass come with it

# the numeric form, for a caller that keeps computing rather than printing
print(round_to(s.mean, s.sd), "vs the stored", s.mean)

## Verdicts and assumptions

`Verdict` is the shared status vocabulary — `identified | downgraded | blocked | unsupported
| unverified` — used by identification, transport, transfer plans, and design methods alike.
Its invariants are enforced at construction: a `blocked` verdict needs a reason, a
`downgraded` one needs at least one named `Assumption`, and an `identified` one cannot carry
an unverified assumption. "Identified, assuming a few things nobody checked" is precisely the
claim this type refuses to let you make.

In [ ]:
overlap = Assumption(
    name="overlap",
    facet="population",
    statement="every target stratum has positive support in the source",
    challenged_by="propensity / dose overlap diagnostic",
)
print(overlap.state)

v = Verdict(status="downgraded", route="backdoor", assumptions=(overlap,))
print(v.status, v.licensed, [a.name for a in v.assumptions])

In [ ]:
refused = []
for label, bad in (
    ('status="blocked"', lambda: Verdict(status="blocked")),
    ('status="downgraded"', lambda: Verdict(status="downgraded")),
    ("identified, assumption unchecked", lambda: Verdict(status="identified", assumptions=(overlap,))),
):
    try:
        bad()
    except ValueError as exc:
        refused.append([label, str(exc)])
table(refused, headers=("construction", "refused because"))

print(Verdict(status="identified", assumptions=(overlap.satisfied(),)).status)
print(Verdict(status="blocked", reason="no admissible adjustment set").licensed)

A `LedgerLine` is one entry in the assumption ledger. It may carry an `Assumption` (something
that could be false) or just record a fact (a unit conversion). `source`/`target` hold the
content hashes of the specs on either side when there are two.

In [ ]:
line = LedgerLine(
    kind="facet:population",
    statement="transferred from region=north to region=all under overlap",
    assumption=overlap.asserted(),
    detail={"from": "north", "to": "all"},
)
print(line.kind, "|", line.assumption.state if line.assumption else None)

## Typed failures

`Unsupported`, `Blocked`, `Unverified` are *returned* by code that must not guess. They are
falsy, need a non-empty reason, and serialize. `is_failure` narrows a `float | Failure`.

The alternative is the `nan` that arithmetics into a report, or the exception caught three
frames up by a handler that logs and continues. Rule 5 exists because four separate
wrong-number bugs in the parent repo trace to exactly that.

In [ ]:
def realize(value_ok: bool) -> float | Failure:
    if not value_ok:
        return Blocked(reason="adjustment set contains an unmeasured variable", detail={"node": "U"})
    return 0.42


rows = []
for ok in (True, False):
    r = realize(ok)
    rows.append([ok, r.status, f"{r.reason} {r.detail}"] if is_failure(r) else [ok, "value", r])
table(rows, headers=("input", "status", "what came back"))

print(bool(Unsupported(reason="no marginal capability", missing=("marginal",))), bool(Unverified(reason="overlap not checked")))

# the three share a field type, not a base class (composition over inheritance)
reason: NonEmptyStr = "a reason that is not blank"
try:
    Blocked(reason="   ")
except Exception as exc:
    print(type(exc).__name__, "- a blank reason is refused")

## Statistics for acceptance tests

Every rate criterion in the roadmap states N and an exact binomial acceptance region at
α = 0.001 (review B5). `clopper_pearson` produces it. This is what stops a coverage gate from
being a coin flip: at n = 40, a *correct* 95% interval procedure fails a naive "≥ 95%" check
often enough to make CI a nuisance, and a broken one passes often enough to survive.

In [ ]:
region: AcceptanceRegion = clopper_pearson(n=500, p=0.05, alpha=0.001)
show(region)
print("count 25 accepted:", region.accepts(25), "| rate bounds:", region.rate_bounds)

In [ ]:
ns = [50, 100, 200, 500, 1000, 2000, 5000]
bounds = [clopper_pearson(n=n, p=0.05, alpha=0.001).rate_bounds for n in ns]
fig = lines(
    ns,
    {"upper accepted rate": [b[1] for b in bounds], "lower accepted rate": [b[0] for b in bounds]},
    colors=(ORANGE, BLUE),
    title="Why every rate criterion states its N",
    subtitle="exact binomial acceptance band around a true rate of 5%, α = 0.001",
    x_title="simulations run", y_title="observed rate",
)
mark_x(fig, 500, text="the roadmap's N")
annotate(fig, 50, bounds[0][1], "at n=50, anything up to 20% passes")
caption(fig, "The band is the honest width of a pass/fail test at that N. Stating '95% "
             "coverage' without it is stating a threshold that the sample size cannot resolve.")

In [ ]:
chains = rng.normal(size=(4, 1000))
ar = np.zeros_like(chains)
for c in range(4):
    for t in range(1, 1000):
        ar[c, t] = 0.8 * ar[c, t - 1] + 0.6 * rng.normal()

print("ESS iid   :", round(effective_sample_size(chains)))
print("ESS AR(1) :", round(effective_sample_size(ar)))
print("MCSE      :", mc_standard_error(ar))
print("z vs ref  :", z_score(value=ar.mean(), reference=0.0, reference_se=mc_standard_error(ar)))

In [ ]:
rho = np.linspace(0.0, 0.95, 20)
ess = []
for r in rho:
    chain = np.zeros((4, 1000))
    for c in range(4):
        for t in range(1, 1000):
            chain[c, t] = r * chain[c, t - 1] + rng.normal() * np.sqrt(1 - r**2)
    ess.append(effective_sample_size(chain))

fig = curve_band(
    rho, ess,
    label="effective draws",
    title="Four thousand draws are not four thousand draws",
    subtitle="effective sample size of 4 chains × 1000 draws, against the autocorrelation in them",
    x_title="lag-1 autocorrelation", y_title="effective sample size",
)
caption(fig, "Every Monte-Carlo standard error in the package divides by this number rather "
             "than by the draw count — which is the difference between an honest error bar "
             "on a sticky chain and a confidently wrong one.")

## Testing many things at once

Three subpackages need the same arithmetic: `diagnose.structure` refuting a graph against
forty implied independences, `diagnose.delivery` checking a balance table, `design.program`
deciding a quarter's readouts. So it lives here, at the bottom, rather than being written
three times.

`holm` controls the family-wise error rate — the chance of **any** false positive — and is
right when the output is a single verdict one lucky test should not flip. `benjamini_hochberg`
controls the false discovery rate — the expected share of the positives that are false — and
is right when the output is a ranked list somebody will work through. `none` leaves them
alone and, because it is a named choice rather than an omission, says so on whatever result
carries it.

In [ ]:
raw = [0.001, 0.012, 0.03, 0.04, 0.2, 0.6]
corrections: list[Multiplicity] = ["none", "holm", "benjamini_hochberg"]
table([[f"{p:.3f}"] + [f"{adjust(raw, c)[i]:.4f}" for c in corrections]
       for i, p in enumerate(raw)],
      headers=("raw p", *corrections), title="six tests, adjusted three ways")
for c in corrections:
    print(f"{c:20s} significant at 0.05: {sum(q < 0.05 for q in adjust(raw, c))} of {len(raw)}")

## What this bought you

A number that leaves this package carries how it was summarized, at a resolution it can
support. A claim carries the assumptions it rests on and refuses to call itself identified
while one is unchecked. A function that could not answer says so in a value you cannot
accidentally multiply. And every rate in the test suite is stated against an N and an exact
acceptance region rather than a hopeful threshold.

`nbs/diagnose/01-sbc-and-coverage.ipynb` is where those acceptance regions become the gate
that decides whether a fit is trustworthy.